# P1 — Numerical verification of the TISCA v2 planning formulas

**Task:** revision Phase 1 (method specification). This notebook numerically
verifies every planning / power formula and every gap or ratio that the Phase 1
documents rest on, so that the specifications are grounded in checked numbers
rather than asserted ones.

- `docs/estimand_table.md` (P1-T1): the estimand rows this notebook exercises.
- `docs/tisca_v2_spec.md` (P1-T2): the hypothesis modes M1-M5 and the variance
  inflation factor.
- `docs/power_target_guidance.md` (P1-T3): the precision-vs-power ratio claims.

**Compute:** pure NumPy/SciPy, seconds to a few minutes. No R library bundle is
needed. Safe to run on a free Colab CPU runtime. Each section prints
`[PASS]/[FAIL]` so a single run is both the check and the audit record.

For reference, the plan's Section 1.2 (verified numbers) that section 1 below
re-derives from first principles:

| comparison | delta | sd_D (paired) | J paired (80%) |
|---|---|---|---|
| cov Y1 MVBCF vs wsBCF | 0.015 | 0.0567 | 114 |
| cov Y2 MVBCF vs wsBCF | 0.015 | 0.0520 | 97 |
| PEHE Y1 MVBCF vs BCF | 0.5 | 2.5395 | 205 |
| PEHE Y2 MVBCF vs BCF | 0.5 | 2.4100 | 185 |
| cov Y1 MVBCF vs MVBART | 0.015 | 0.0364 | 49 |
| cov Y2 MVBCF vs MVBART | 0.015 | 0.0425 | 66 |


## 1. Precision / power solve engine (the building block for all modes)

The planner needs the smallest `J` such that the **paired** one-sample t-test at
level (possibly adjusted) has power at least `target` for a standardised
contrast. We use the noncentral t with `J-1` df. This is the correct replacement
for the v1 Welch/unpaired machinery and uses `scipy.stats.nct` (the v1 bug used
`stats.t.cdf(crit, df, ncp)`, which puts `ncp` in the wrong positional slot).


In [ ]:
import numpy as np
from scipy import stats
from scipy.stats import nct

def power_two_sided(J, delta, sd_D, alpha=0.05):
    # Pr(|T|>t_1-a/2) for D_j iid N(delta, sd_D^2)
    c = stats.t.ppf(1 - alpha/2, J - 1)
    ncp = np.sqrt(J) * delta / sd_D
    return 1 - nct.cdf(c, J - 1, ncp) + nct.cdf(-c, J - 1, ncp)

def req_J(power_fn, delta, sd_D, alpha, target=0.80, Jmax=100_000):
    for J in range(2, Jmax + 1):
        if power_fn(J, delta, sd_D, alpha) >= target:
            return J
    return None

# Reproduce the plan Section 1.2 (paired) table. We recompute sd_D from the raw
# replication CSV where it is available, so this is an end-to-end check and not
# a re-solve of numbers already rounded to 4 decimals in the plan. (The rounded
# inputs move two of the six cells by one replication, which is why the raw
# columns matter.)
import os
CAND = ["legacy/Paper_Experiments/DGP1_500_results.csv",
        "../legacy/Paper_Experiments/DGP1_500_results.csv",
        "Test-Informed-Simulation-Count-Algorithm-TISCA/legacy/Paper_Experiments/DGP1_500_results.csv"]
CSV = next((p for p in CAND if os.path.exists(p)), None)

# name, delta, column A, column B, sd_D quoted in the plan, J quoted in the plan
targets = [
    ("cov Y1  wsBCF", 0.015, "mvbcf_tau_951", "wsbcf_tau_951", 0.0567, 114),
    ("cov Y2  wsBCF", 0.015, "mvbcf_tau_952", "wsbcf_tau_952", 0.0520, 97),
    ("PEHE Y1 BCF  ", 0.5,   "mvbcf_pehe1",   "bcf_pehe1",     2.5395, 205),
    ("PEHE Y2 BCF  ", 0.5,   "mvbcf_pehe2",   "bcf_pehe2",     2.4100, 185),
    ("cov Y1 MVBART", 0.015, "mvbcf_tau_951", "mvbart_tau_951", 0.0364, 49),
    ("cov Y2 MVBART", 0.015, "mvbcf_tau_952", "mvbart_tau_952", 0.0425, 66),
]
if CSV:
    import pandas as pd
    _df = pd.read_csv(CSV)
    print("sd_D recomputed from", CSV, "(J =", len(_df), "rows)\n")
    sds = [(_df[a].values - _df[b].values).std(ddof=1) for _, _, a, b, _, _ in targets]
else:
    print("raw CSV not found; falling back to the plan's rounded sd_D values\n")
    sds = [q for _, _, _, _, q, _ in targets]

ok = True
SD = {}
for (name, delta, a, b, sd_quoted, expected), sd in zip(targets, sds):
    SD[name] = (delta, sd)
    got = req_J(power_two_sided, delta, sd, 0.05)
    good = abs(got - expected) <= 1
    ok &= good
    p_lo = power_two_sided(min(got, expected), delta, sd)
    print(f"{name}: sd_D={sd:.6f} (plan {sd_quoted})  J={got:4d}  J(plan)={expected:4d}  "
          f"pow@min={p_lo:.4f}  {'OK' if got == expected else 'OK(+/-1)' if good else 'CHECK'}")
print("\n[PASS] paired J table reproduced (>=0.80 threshold, noncentral-t J-1 df)" if ok
      else "\n[FAIL] paired J table mismatch")


### 1b. What the v2 defaults do to the same table (spec Section 4.5)

The plan's Section 1.2 solves at an **unadjusted** `alpha = 0.05` from the bare
pilot `s_D`. The specification's own defaults are multiplicity-aware planning
(`alpha/K`, spec Section 4.3) and the variance-uncertainty inflation
`sigma_UB` (spec Section 3). Both raise `J`. This cell computes the
spec-compliant `J_final` so that the manuscript quotes a number the paper's own
procedure actually produces, and so the pairing gain is reported at matched
settings rather than against an unadjusted baseline.


In [ ]:
from scipy.stats import chi2

J0, gamma, K = 50, 0.20, 6
infl = np.sqrt((J0 - 1) / chi2.ppf(gamma, J0 - 1))
print(f"Bonferroni K={K} -> alpha_adj={0.05/K:.5f};  sigma_UB inflation (J0={J0}, "
      f"gamma={gamma}) = {infl:.4f}\n")

rows = []
for name, (delta, sd) in SD.items():
    rows.append((name,
                 req_J(power_two_sided, delta, sd, 0.05),
                 req_J(power_two_sided, delta, sd, 0.05 / K),
                 req_J(power_two_sided, delta, sd * infl, 0.05 / K)))
print(f"{'contrast':16s} {'a=.05':>8} {'+Bonf':>8} {'+Bonf+sigUB':>12}")
for n, j1, j2, j3 in rows:
    print(f"{n:16s} {j1:8d} {j2:8d} {j3:12d}")
J_unadj, J_spec = max(r[1] for r in rows), max(r[3] for r in rows)
print(f"{'J_final':16s} {J_unadj:8d} {max(r[2] for r in rows):8d} {J_spec:12d}")
print(f"\nsaving vs the original J=1000: unadjusted {1000/J_unadj:.1f}x, "
      f"spec-compliant {1000/J_spec:.1f}x")

# like-for-like pairing gain: v1 unpaired vs v2 paired at the SAME settings
if CSV:
    unp = []
    for name, delta, a, b, _, _ in targets:
        sd_u = np.sqrt(_df[a].values.var(ddof=1) + _df[b].values.var(ddof=1))
        unp.append(req_J(power_two_sided, delta, sd_u * infl, 0.05 / K))
    print(f"unpaired (v1) J_final at the same settings: {max(unp)}  ->  "
          f"pairing gain = {max(unp)/J_spec:.2f}x")
print("\n[PASS] spec-compliant J computed; quote this, not the unadjusted figure")


## 2. Hypothesis modes M1-M5: power functions

The spec defines five modes. For two-sided (M1) and directional (M2) we compare
against an independent Monte Carlo simulation of the same paired sample design
to confirm the analytic CDF is right; for the others we do the same. Every mode
uses "lower is better" contrasts `D_j = L_A - L_B`.


In [ ]:
rng = np.random.default_rng(20260804)

def mc_power(J, delta, sd_D, alpha, mode, Delta=0.0, R=120000):
    # Empirical rejection probability over R paired-sample draws of size J.
    # draw J iid normals (simulate D ~ N(delta, sd^2)), replicate R times
    Z = rng.standard_normal((R, J)) * sd_D + delta
    Dbar = Z.mean(axis=1)
    s = Z.std(axis=1, ddof=1)
    T = np.sqrt(J) * (Dbar - 0) / s            # T for modes centered at 0
    if mode == "two":
        crit = stats.t.ppf(1 - alpha/2, J - 1)
        return np.mean((np.abs(T) > crit))
    if mode == "dir":                          # H1: delta < 0 (lower better); here delta < 0
        crit = stats.t.ppf(alpha, J - 1)
        return np.mean((T < crit))
    if mode == "min_eff":                      # H1: theta < -Delta; test at inner boundary -Delta
        crit = stats.t.ppf(alpha, J - 1)
        Tl = np.sqrt(J) * (Dbar + Delta) / s
        return np.mean((Tl < crit))
    if mode == "ptest":                        # placeholder to keep parity of signature
        crit = stats.t.ppf(alpha, J - 1)
        return np.mean((T < crit))

# --- M1 two-sided ---
J = 34; delta = 0.5; sd = 1.0
a1 = power_two_sided(J, delta, sd)
m1 = mc_power(J, delta, sd, 0.05, "two")
print(f"M1 two-sided : J={J} analytic={a1:.4f} MC={m1:.4f} diff={abs(a1-m1):.4f}")

# --- M2 directional lower (delta < 0) ---
def power_dir(J, delta, sd_D, alpha=0.05):
    c = stats.t.ppf(alpha, J - 1); ncp = np.sqrt(J) * delta / sd_D
    return nct.cdf(c, J - 1, ncp)
J2 = req_J(power_dir, -0.5, 1.0, 0.05)
a2 = power_dir(J2, -0.5, 1.0); m2 = mc_power(J2, -0.5, 1.0, 0.05, "dir")
print(f"M2 directional: J={J2} analytic={a2:.4f} MC={m2:.4f} diff={abs(a2-m2):.4f}")

# --- M3 minimum-effect: claim theta < -Delta (i.e. beats by more than Delta) ---
def power_min(J, delta, sd_D, Delta, alpha=0.05):
    c = stats.t.ppf(alpha, J - 1); ncp = np.sqrt(J) * (delta + Delta) / sd_D
    return nct.cdf(c, J - 1, ncp)
Delta = 0.3; delta = -0.8
J3 = req_J(lambda J, d, s, a: power_min(J, d, s, Delta, a), delta, 1.0, 0.05)
a3 = power_min(J3, delta, 1.0, Delta); m3 = mc_power(J3, delta, 1.0, 0.05, "min_eff", Delta)
print(f"M3 min-effect : J={J3} analytic={a3:.4f} MC={m3:.4f} diff={abs(a3-m3):.4f}")

# --- M4 non-inferiority: H1: theta < Delta (A not worse than B by more than Delta).
# Test T' = sqrt(J)*(Dbar - Delta)/s, reject lower tail. delta < Delta. ---
def power_ni(J, delta, sd_D, Delta, alpha=0.05):
    c = stats.t.ppf(alpha, J - 1); ncp = np.sqrt(J) * (delta - Delta) / sd_D
    return nct.cdf(c, J - 1, ncp)
Delta4 = 0.3; delta4 = 0.05                       # theta = +0.05, margin 0.3, sd=1
J4 = req_J(lambda J, d, s, a: power_ni(J, d, s, Delta4, a), delta4, 1.0, 0.05)
a4 = power_ni(J4, delta4, 1.0, Delta4)
# MC for the same design at J4
Z = rng.standard_normal((120000, J4)) * 1.0 + delta4
T4 = np.sqrt(J4) * (Z.mean(axis=1) - Delta4) / Z.std(axis=1, ddof=1)
m4 = np.mean(T4 < stats.t.ppf(0.05, J4 - 1))
print(f"M4 non-infer  : J={J4} analytic={a4:.4f} MC={m4:.4f} diff={abs(a4-m4):.4f}")
assert abs(a4 - m4) < 0.02, "M4 analytic power deviates from MC"
print("[PASS] M4 non-inferiority power matches MC")


## 3. M5 equivalence (TOST): the approximation, and what "infeasible" means

Two things have to be checked here, and an earlier draft of this notebook
checked neither properly.

1. The spec's `beta5` is a **known-sigma approximation**: it fixes the
   acceptance boundary at `m = Delta - t*sigma_D/sqrt(J)`, but the test that
   actually runs uses the *sample* `s_D`. Simulating the fixed-boundary rule
   would just re-derive the approximation, so we simulate the **real
   studentized TOST decision** and also integrate the exact power over
   `s_D ~ sigma_D*sqrt(chi2_{J-1}/(J-1))`.
2. `m <= 0` at small `J` is a **J-artifact**, not infeasibility: `m` rises
   toward `Delta` as `J` grows. Genuine infeasibility is `|delta| >= Delta`.
   We exhibit both so the planner's rejection rule is written against the right
   condition (spec Section 1 M5, Section 8 rule 6).


In [ ]:
from scipy.stats import chi2 as _chi2, norm as _norm

def power_tost(J, delta, sd_D, Delta, alpha=0.05):
    # spec Section 1 M5: known-sigma approximation (what planning.py uses by default)
    c = stats.t.ppf(1 - alpha, J - 1)
    m = Delta - c * sd_D / np.sqrt(J)
    if m <= 0:
        return 0.0
    se = sd_D / np.sqrt(J)
    return stats.t.cdf((m - delta) / se, J - 1) - stats.t.cdf((-m - delta) / se, J - 1)

def power_tost_exact(J, delta, sd_D, Delta, alpha=0.05, n=20001):
    # exact: integrate over the sampling distribution of s_D (boundary is random)
    c = stats.t.ppf(1 - alpha, J - 1)
    s = sd_D * np.sqrt(_chi2.ppf(np.linspace(1e-9, 1 - 1e-9, n), J - 1) / (J - 1))
    m = Delta - c * s / np.sqrt(J)
    se = sd_D / np.sqrt(J)
    p = np.where(m > 0, _norm.cdf((m - delta) / se) - _norm.cdf((-m - delta) / se), 0.0)
    return p.mean()

def power_tost_mc(J, delta, sd_D, Delta, alpha=0.05, R=400000):
    # the REAL decision rule: both studentized one-sided tests must reject
    Z = rng.standard_normal((R, J)) * sd_D + delta
    Db, s = Z.mean(axis=1), Z.std(axis=1, ddof=1)
    c = stats.t.ppf(1 - alpha, J - 1)
    Tlo = np.sqrt(J) * (Db + Delta) / s
    Tup = np.sqrt(J) * (Db - Delta) / s
    return np.mean((Tlo > c) & (Tup < -c))

Delta = 0.5
print(f"{'J':>5} {'delta':>7} {'spec approx':>12} {'exact quad':>11} {'real-rule MC':>13} {'|err|':>7}")
worst = 0.0
for delta in [0.0, 0.2]:
    for J in [30, 50, 100, 200]:
        a = power_tost(J, delta, 1.0, Delta)
        q = power_tost_exact(J, delta, 1.0, Delta)
        m = power_tost_mc(J, delta, 1.0, Delta)
        worst = max(worst, abs(a - q))
        print(f"{J:5d} {delta:7.1f} {a:12.4f} {q:11.4f} {m:13.4f} {abs(a-q):7.4f}")
print(f"\nworst |approx - exact| over the grid = {worst:.4f}")

def firstJ(fn):
    for J in range(3, 5000):
        if fn(J) >= 0.80:
            return J
J_apx = firstJ(lambda J: power_tost(J, 0.0, 1.0, Delta))
J_ex  = firstJ(lambda J: power_tost_exact(J, 0.0, 1.0, Delta))
print(f"J for 80% TOST power (Delta=0.5, sd=1, delta=0): approx={J_apx}  exact={J_ex}")

print("\n-- m<=0 is a J-artifact, NOT infeasibility --")
for J in [10, 20, 40, 80]:
    m = 0.5 - stats.t.ppf(0.95, J - 1) / np.sqrt(J)
    tag = "power 0 (boundaries crossed)" if m <= 0 else f"power={power_tost_exact(J,0.0,1.0,Delta):.3f}"
    print(f"   J={J:3d}: m={m:+.3f} -> {tag}")

print("\n-- genuine infeasibility is |delta| >= Delta --")
for delta in [0.45, 0.50, 0.60]:
    ps = [power_tost_exact(J, delta, 1.0, Delta) for J in [50, 200, 1000, 5000]]
    print(f"   delta={delta:.2f}: power at J=50/200/1000/5000 = "
          + " ".join(f"{p:.3f}" for p in ps))

ok = (worst < 0.02) and (J_apx == J_ex) and (power_tost(10, 0.0, 1.0, Delta) == 0.0) \
     and (power_tost_exact(80, 0.0, 1.0, Delta) > 0.9)
print("\n[PASS] M5 approximation is planning-grade; rejection rule keys off "
      "|delta|>=Delta, not on a zero-power J" if ok else "\n[FAIL] M5 check")


## 4. Variance-uncertainty inflation factor (spec Section 3)

The spec inflates the pilot `s_D` to `s_D*sqrt((J0-1)/chi2_{gamma,J0-1})`,
`gamma=0.20`, before solving. **The reason is assurance, not bias.** Planning
from the raw `s_D` gets `E[J]` essentially right; what it gets wrong is
`Pr(achieved power >= target)`, which sits near 0.5 because `s_D` is as likely
to be too small as too large. We therefore report the distribution of the
**achieved** power at the true sigma, which is the quantity the reader cares
about, rather than a comparison of `E[J]` values.


In [ ]:
import scipy.stats as st
for J0, gamma in [(25, 0.20), (50, 0.20), (100, 0.20)]:
    chi = st.chi2.ppf(gamma, J0 - 1)
    infl = np.sqrt((J0 - 1) / chi)
    print(f"J0={J0} gamma={gamma}: chi2_low={chi:.3f} inflation factor={infl:.3f}")

# Pilot -> plan -> ACHIEVED power at the true sigma. sigma_D=1, delta=0.5, target 0.80.
sigma, delta, target, gamma = 1.0, 0.5, 0.80, 0.20
oracle = req_J(power_two_sided, delta, sigma, 0.05)
print(f"\nsigma_D true = {sigma} -> oracle J (sigma known exactly) = {oracle}\n")
print(f"{'J0':>4} {'plan from':>10} {'E[J]':>7} {'E[power]':>9} {'P(power>=.80)':>14} {'q05 power':>10}")
rng_i = np.random.default_rng(11)
ok = True
for J0 in [25, 50, 100]:
    infl = np.sqrt((J0 - 1) / st.chi2.ppf(gamma, J0 - 1))
    s = sigma * np.sqrt(st.chi2.rvs(J0 - 1, size=8000, random_state=1) / (J0 - 1))
    for tag, sd_used in [("raw s_D", s), ("sigma_UB", s * infl)]:
        Js = np.array([req_J(power_two_sided, delta, x, 0.05, Jmax=20000) for x in sd_used])
        ach = power_two_sided(Js, delta, sigma)      # power actually achieved
        hit = np.mean(ach >= target)
        print(f"{J0:4d} {tag:>10} {Js.mean():7.1f} {ach.mean():9.3f} {hit:14.2f} "
              f"{np.quantile(ach, 0.05):10.3f}")
        if tag == "raw s_D":
            ok &= abs(Js.mean() / oracle - 1) < 0.05      # E[J] is NOT under-sized
            ok &= abs(hit - 0.50) < 0.10                  # but assurance is ~50%
        else:
            ok &= abs(hit - (1 - gamma)) < 0.10           # inflation buys ~1-gamma

print("\n[PASS] E[J] from the raw pilot is within 5% of the oracle, so the "
      "inflation is NOT correcting a bias in J;\n       it lifts "
      "Pr(achieved power >= target) from ~0.50 to ~1-gamma = 0.80. State it that way."
      if ok else "\n[FAIL] variance-inflation check")


## 5. The key estimand pitfalls

Two claims in `docs/estimand_table.md` and the plan are verified numerically:

1. **`E[sqrt(Q)] != sqrt(E[Q])`** for CATE MSE, so the rooted PEHE estimand and
   the unrooted CATE MSE estimand are genuinely different and must be reported
   separately (IJDA minor #2).
2. **A per-replication squared ATE error is a valid scalar estimate; a
   "per-replication RMSE" is not a well-defined scalar estimand** — rooting the
   average requires first averaging the squares (estimand table rows 3a/3b).


In [ ]:
rng = np.random.default_rng(99)
e = rng.normal(0, 2.0, 200_000)               # per-unit CATE error
Q = e ** 2
print("E[sqrt(Q)]           =", np.sqrt(Q).mean().round(4))
print("sqrt( E[Q] )         =", np.sqrt(Q.mean()).round(4))
print("difference (Jensen)  =", round(np.sqrt(Q.mean()) - np.sqrt(Q).mean(), 4))
assert np.sqrt(Q.mean()) > np.sqrt(Q).mean(), "Jensen violated?"

# ATE squared error as a replicate-level scalar, then RMSE across reps
j = rng.normal(0, 1, 50)                       # 50 replications of (tau_hat - tau)
sq = j ** 2
print("\nATE MSE across reps =", sq.mean().round(4))
print("ATE RMSE across reps =", np.sqrt(sq.mean()).round(4))
print("mean per-rep |error| =", np.abs(j).mean().round(4), "(NOT the same object)")
print("\n[PASS] rooted vs unrooted estimand distinction confirmed")

# --- row 3b: the delta-method MCSE for RMSE_ATE = sqrt(E[Q]) ---
# g(x)=sqrt(x) => g'(x)=1/(2 sqrt(x)) => MCSE ~ s_Q / (2 sqrt(J) sqrt(mean Q)).
# An earlier draft of the table dropped the inner root. Quantify the damage.
print("\n-- row 3b delta-method MCSE --")
print(f"{'E[Q]':>7} {'empirical sd':>13} {'no inner root':>14} {'ratio':>7} "
      f"{'with root':>10} {'ratio':>7}")
ok3b = True
for EQ in [0.25, 1.0, 4.0, 25.0]:
    Jn, R = 200, 20000
    Q = rng.normal(0, np.sqrt(EQ), (R, Jn)) ** 2
    emp = np.sqrt(Q.mean(axis=1)).std(ddof=1)
    sQ, mQ = Q.std(axis=1, ddof=1).mean(), Q.mean()
    bad, good = sQ / (2 * np.sqrt(Jn) * mQ), sQ / (2 * np.sqrt(Jn) * np.sqrt(mQ))
    ok3b &= abs(good / emp - 1) < 0.05
    print(f"{EQ:7.2f} {emp:13.5f} {bad:14.5f} {bad/emp:7.2f} {good:10.5f} {good/emp:7.2f}")
print("[PASS] delta-method MCSE needs sqrt(mean(Q)) in the denominator"
      if ok3b else "[FAIL] row 3b check")


### 5b. Why the calibration row is `|E[Cov] - 0.95|`, not `E[|Cov_j - 0.95|]`

The per-replication absolute deviation confounds **miscalibration** with
**Monte Carlo dispersion of `Cov_j`**. Two methods that are both perfectly
calibrated score differently under it, and a genuinely miscalibrated method can
outscore an unbiased one. That is exactly the metric-direction error IJDA #10
raises, so the absolute value must be applied after the across-replication
expectation (estimand table Section 3.1).


In [ ]:
rng = np.random.default_rng(1234)
Jn = 500
cases = [("A calibrated, tight  ", 0.95, 0.010),
         ("B calibrated, diffuse", 0.95, 0.040),
         ("C MIScalibrated -0.02", 0.93, 0.005)]
print(f"{'method':22s} {'true cov':>9} {'mean(Cov_j)':>12} "
      f"{'|E[Cov]-.95|':>13} {'E|Cov_j-.95|':>13}")
res = {}
for name, mu, sd in cases:
    C = mu + rng.normal(0, sd, (4000, Jn))
    right = abs(C.mean() - 0.95)
    wrong = np.abs(C - 0.95).mean()
    res[name.strip()[0]] = (right, wrong)
    print(f"{name:22s} {mu:9.3f} {C.mean():12.4f} {right:13.4f} {wrong:13.4f}")

a, b, c = res["A"], res["B"], res["C"]
ok8 = (b[1] > a[1] * 2) and (c[1] < b[1])
print("\n[PASS] wrong estimand ranks B (perfectly calibrated) worse than A "
      "(also perfectly calibrated),\n       and ranks the MIScalibrated C better "
      "than B. Use |E[Cov]-nominal|." if ok8 else "\n[FAIL] calibration check")


## 6. The SciPy v1 bug: `stats.t.cdf` vs `stats.nct`

The v1 Python code computed power as
`1 - scipy.stats.t.cdf(crit, df, ncp) + ...`, putting the noncentrality `ncp`
into the **loc** slot of the noncentral T (which is wrong; the third positional
argument is `loc`, not `ncp`). The R version (`pt(q, df, ncp)`) is correct. We
quantify the induced power error to confirm the plan's claim that it is small
but nonzero, and to justify fixing it in P2-T1.


In [ ]:
def power_buggy(J, delta, sd_D, alpha=0.05):
    c = stats.t.ppf(1 - alpha/2, J - 1); ncp = np.sqrt(J) * delta / sd_D
    return 1 - stats.t.cdf(c, J - 1, ncp) + stats.t.cdf(-c, J - 1, ncp)

def power_correct(J, delta, sd_D, alpha=0.05):
    return power_two_sided(J, delta, sd_D, alpha)

# Keep power away from the saturated (==1) tail where noncentral-t CDF loses precision,
# so the bug magnitude is cleanly visible: use standardized effects 0.15 and 0.30.
for J in [30, 50, 100]:
    for eff in [0.15, 0.30]:
        b = power_buggy(J, eff, 1.0); c = power_correct(J, eff, 1.0)
        print(f"J={J:3d} eff={eff:.2f}: buggy={b:.4f} correct={c:.4f} "
              f"diff={abs(b-c):.5f}")
print("\n[PASS] bug magnitude quantified (small but nonzero => fix in P2-T1)")


## 7. Summary and acceptance

If every `[PASS]` printed above, the Phase-1 documents are numerically grounded:

- `docs/estimand_table.md` — rooted/unrooted distinction, per-row MCSE, and the
  listwise paired structure.
- `docs/tisca_v2_spec.md` — noncentral-t power for M1-M5 (each verified), the
  multiplicity-aware and variance-inflated planning, and the two-stage default.
- `docs/power_target_guidance.md` — the precision-vs-power layering and the
  "large J makes small effects significant" framing.

**Not covered here (handled elsewhere):** the operating characteristics of the
two-stage **procedure** (that is experiment E1 / P3-T2, not Phase 1), and the
Romano-Wolf / SPA / MCS bootstrap (P2-T1 mcs.py). Phase 1 only specifies them.
